In [12]:
import pandas as pd
import snowflake.connector
import json
import re

In [5]:
conn = snowflake.connector.connect(
    account="VSB79059-HJB86910",
    user="TYLER.HAAS@INNOVATION.CA.GOV",
    authenticator="externalbrowser",
    role="TRANSFORMER_ENGCA_DEV",
)

cur = conn.cursor()
cur.execute("""
    SELECT
        *
    FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS s
    WHERE PUBLICATION_STATUS = 'published'
""")

df = cur.fetch_pandas_all()
df.reset_index(names="temp_id", inplace=True)

In [6]:
df.head()

,temp_id,SURVEY_RESPONDENT_ID,SURVEY_ID,AGE,GENDER_ARRAY,GENDER_CATEGORY,RACE_ETHNICITY_ARRAY,RACE_ETHNICITY_CATEGORY,USER_STATUS,PUBLICATION_STATUS,...,ROLE_AT_WORK,COUNTY,REGION,FIELD_OF_WORK,ECONOMIC_IMPACT_EXPECTATION,GOVERNMENT_ACTION_SUGGESTION,PERSONAL_AI_IMPACT,AVAILABILITY_FOR_DISCUSSION,FIELDS_COMPLETED_COUNT,_LOADED_AT
0,0,44296481-115b-41de-8f2a-feffefc24e46,5e1d74ff-9414-4ac5-865b-bde8aa02798d,Over 65,"[\n ""Man""\n]",Man (only),None,None,active,published,...,I don't currently work,Sonoma County,Far North / North Coast,Legal,The production of physical products should bec...,Basically get out of the way. Government shoul...,"Being retired, not much impact.",Maybe,10,2026-05-21 11:15:40.803488-07:00
1,1,b42ca0a9-9ee9-4a07-ad5c-ee03b3daefd9,2db00d1d-1299-4760-95ff-2b5b7850a5a7,25-44,"[\n ""I don't want to say""\n]",I don't want to say (only),None,None,active,published,...,I don't want to say,Kern County,Central Valley,I don't want to say,None,None,None,No,7,2026-05-21 11:15:40.803488-07:00
2,2,2242ff96-c6a1-4669-8309-93eaa4a2fa09,b78cbf7d-a508-4812-ba39-d0fd060d06a9,None,None,None,None,None,active,published,...,"Contractor, freelancer, or gig worker",Solano County,Bay Area,Utilities or waste management,Overall? They'll make it worse due to taking j...,"Do not allow AI to work, even for free.",None,No,8,2026-05-21 11:15:40.803488-07:00
3,3,e99183f9-704d-4475-bcfc-36d770b5866c,6cf926ac-34a6-48c4-8a56-fc78bac7a26a,45-64,None,None,None,None,active,published,...,Employee (non-management),Sacramento County,Sacramento Valley / Sierra Foothills,Government,None,None,None,Maybe,6,2026-05-21 11:15:40.803488-07:00
4,4,799530e7-2104-4fe6-9fed-f09b375fe034,e053ccc3-07e8-4794-b67c-944837c1113f,45-64,"[\n ""Woman""\n]",Woman (only),"[\n ""White""\n]",White (only),active,published,...,Employee (non-management),Alameda County,Bay Area,Healthcare,If things continue on their current trajectory...,I absolutely believe that government needs to ...,I am a clinical psychologist. Though I do not ...,Yes,11,2026-05-21 11:15:40.803488-07:00


In [7]:
OVERALL_AI_SENTIMENT_COLS = ["ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT"]
GOVERNMENT_ACTION_SENTIMENT_COLS = ["GOVERNMENT_ACTION_SUGGESTION"]

SENTIMENT_OPTIONS = ["PRO", "ANTI", "NEUTRAL"]

## Step 2: Zero-Shot Classification

In [8]:
MODEL = "claude-4-sonnet"

OVERALL_SYSTEM_PROMPT = """
You are classifying California residents' survey responses about AI.
Classify the respondent's policy position and attitude toward AI as a technology — not the emotional tone of their writing.

PRO: The respondent views AI as broadly beneficial and supports its development, adoption, or expansion.
ANTI: The respondent views AI as broadly harmful and opposes its development, adoption, or expansion.
NEUTRAL: The respondent is ambivalent, holds mixed views, or does not express a clear directional stance.

Respond with JSON only, no markdown: {"label": "pro|anti|neutral", "rationale": "one sentence"}
""".strip()

GOVERNMENT_SYSTEM_PROMPT = """
You are classifying California residents' survey responses about AI governance.
Classify the respondent's policy position on government action regarding AI — not the emotional tone of their writing.

PRO: The respondent supports active government involvement (regulation, oversight, restrictions, public investment, or standards-setting).
ANTI: The respondent opposes government involvement and prefers a market-driven or hands-off approach.
NEUTRAL: The respondent is ambivalent, proposes a balanced approach, or does not express a clear directional stance.

Respond with JSON only, no markdown: {"label": "pro|anti|neutral", "rationale": "one sentence"}
""".strip()

In [9]:
def esc(s):
    # Escape for safe embedding in a Snowflake SQL string literal
    return s.replace("'", "''").replace("\n", " ").replace("\r", "")

sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    CASE
        WHEN COALESCE(ECONOMIC_IMPACT_EXPECTATION, '') != ''
          OR COALESCE(PERSONAL_AI_IMPACT, '') != ''
        THEN SNOWFLAKE.CORTEX.COMPLETE(
            '{MODEL}',
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(OVERALL_SYSTEM_PROMPT)}'),
                OBJECT_CONSTRUCT('role', 'user', 'content',
                    CONCAT(
                        'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                        ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)')
                    )
                )
            ),
            OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
        )
    END AS OVERALL_RAW,
    CASE
        WHEN COALESCE(GOVERNMENT_ACTION_SUGGESTION, '') != ''
        THEN SNOWFLAKE.CORTEX.COMPLETE(
            '{MODEL}',
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(GOVERNMENT_SYSTEM_PROMPT)}'),
                OBJECT_CONSTRUCT('role', 'user', 'content', COALESCE(GOVERNMENT_ACTION_SUGGESTION, ''))
            ),
            OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
        )
    END AS GOVERNMENT_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
"""

cur = conn.cursor()
cur.execute(sql)
raw_df = cur.fetch_pandas_all()

In [13]:
raw_df.head()

,SURVEY_RESPONDENT_ID,OVERALL_RAW,GOVERNMENT_RAW,overall_parsed,government_parsed
0,44296481-115b-41de-8f2a-feffefc24e46,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{...","{'label': 'error', 'rationale': 'name 'json' i...","{'label': 'error', 'rationale': 'name 'json' i..."
1,b42ca0a9-9ee9-4a07-ad5c-ee03b3daefd9,None,None,None,None
2,2242ff96-c6a1-4669-8309-93eaa4a2fa09,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{...","{'label': 'error', 'rationale': 'name 'json' i...","{'label': 'error', 'rationale': 'name 'json' i..."
3,e99183f9-704d-4475-bcfc-36d770b5866c,None,None,None,None
4,799530e7-2104-4fe6-9fed-f09b375fe034,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{...","{'label': 'error', 'rationale': 'name 'json' i...","{'label': 'error', 'rationale': 'name 'json' i..."


In [14]:
def parse_cortex_response(raw):
    if raw is None:
        return pd.Series({"label": None, "rationale": None})
    try:
        content = json.loads(raw)["choices"][0]["messages"].strip()
        content = re.sub(r"^```(?:json)?\s*", "", content).rstrip("` \n")
        result = json.loads(content)
        return pd.Series({"label": result.get("label", "error").lower(), "rationale": result.get("rationale", "")})
    except Exception as e:
        return pd.Series({"label": "error", "rationale": str(e)})

zero_shot_df = raw_df[["SURVEY_RESPONDENT_ID"]].copy()
zero_shot_df[["overall_ai_sentiment", "overall_rationale"]] = raw_df["OVERALL_RAW"].apply(parse_cortex_response)
zero_shot_df[["government_action_sentiment", "government_rationale"]] = raw_df["GOVERNMENT_RAW"].apply(parse_cortex_response)

zero_shot_df.head()

,SURVEY_RESPONDENT_ID,overall_ai_sentiment,overall_rationale,government_action_sentiment,government_rationale
0,44296481-115b-41de-8f2a-feffefc24e46,neutral,The respondent provides a balanced analysis of...,anti,The respondent advocates for minimal governmen...
1,b42ca0a9-9ee9-4a07-ad5c-ee03b3daefd9,None,None,None,None
2,2242ff96-c6a1-4669-8309-93eaa4a2fa09,anti,The respondent expresses concern that AI will ...,pro,The respondent advocates for a complete prohib...
3,e99183f9-704d-4475-bcfc-36d770b5866c,None,None,None,None
4,799530e7-2104-4fe6-9fed-f09b375fe034,anti,The respondent expresses consistent opposition...,pro,The respondent strongly advocates for extensiv...


## Zeroshot Labelling Results

In [15]:
labeled_df = df.merge(zero_shot_df, on="SURVEY_RESPONDENT_ID", how="left")

audit_cols = [
    "SURVEY_RESPONDENT_ID",
    "ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT",
    "overall_ai_sentiment", "overall_rationale",
    "GOVERNMENT_ACTION_SUGGESTION",
    "government_action_sentiment", "government_rationale",
]
labeled_df[audit_cols].head(10)

,SURVEY_RESPONDENT_ID,ECONOMIC_IMPACT_EXPECTATION,PERSONAL_AI_IMPACT,overall_ai_sentiment,overall_rationale,GOVERNMENT_ACTION_SUGGESTION,government_action_sentiment,government_rationale
0,44296481-115b-41de-8f2a-feffefc24e46,The production of physical products should bec...,"Being retired, not much impact.",neutral,The respondent provides a balanced analysis of...,Basically get out of the way. Government shoul...,anti,The respondent advocates for minimal governmen...
1,b42ca0a9-9ee9-4a07-ad5c-ee03b3daefd9,None,None,None,None,None,None,None
2,2242ff96-c6a1-4669-8309-93eaa4a2fa09,Overall? They'll make it worse due to taking j...,None,anti,The respondent expresses concern that AI will ...,"Do not allow AI to work, even for free.",pro,The respondent advocates for a complete prohib...
3,e99183f9-704d-4475-bcfc-36d770b5866c,None,None,None,None,None,None,None
4,799530e7-2104-4fe6-9fed-f09b375fe034,If things continue on their current trajectory...,I am a clinical psychologist. Though I do not ...,anti,The respondent expresses consistent opposition...,I absolutely believe that government needs to ...,pro,The respondent strongly advocates for extensiv...
5,e5c54257-28c5-4285-91e8-aab9b6349dbc,AI will automate enormous portions of the econ...,AI has affected my workplace profoundly. I'm a...,anti,The respondent views AI as harmful due to its ...,California should require AI companies to cont...,pro,The respondent advocates for mandatory governm...
6,002239f4-01ca-4dc8-b0db-002ecc6665b5,Badly. It is a waste of energy/water and gener...,Increased negative attitudes toward soft skill...,anti,The respondent views AI as economically wastef...,Regulate genAI usage according to recommendati...,pro,The respondent explicitly supports government ...
7,9d785e32-8d45-4d1d-9a70-d2aa91c5d9fb,"In some cases, people are relying too much on ...",There’s good and bad if you have a technical p...,neutral,The respondent acknowledges both positive and ...,I believe in order for us to be able to streng...,neutral,The respondent expresses a general goal about ...
8,6adf4967-dd49-4f7f-ba35-8bba6ff97535,AI shifts the power dynamic from labor to capi...,"AI has not infiltrated government as much yet,...",anti,The respondent expresses clear opposition to A...,Regulate AI. Make sure models are being design...,pro,The respondent explicitly calls for AI regulat...
9,5ab5b645-0a71-4925-93ef-539bd9a8a865,Mass disruption of the workforce resulting in ...,I work in the cyber-security industry as an En...,neutral,The respondent acknowledges both negative econ...,Ensure displaced employees are able to receive...,pro,The respondent supports active government inte...


In [20]:
print(labeled_df['overall_ai_sentiment'].value_counts(normalize=True))
print("\n")
print(labeled_df['government_action_sentiment'].value_counts(normalize=True))

overall_ai_sentiment
anti       0.576786
neutral    0.265179
pro        0.158036
Name: proportion, dtype: float64


government_action_sentiment
pro        0.894981
neutral    0.060409
anti       0.044610
Name: proportion, dtype: float64


In [22]:
print(labeled_df['overall_ai_sentiment'].value_counts())
print(labeled_df['government_action_sentiment'].value_counts())

overall_ai_sentiment
anti       646
neutral    297
pro        177
Name: count, dtype: int64
government_action_sentiment
pro        963
neutral     65
anti        48
Name: count, dtype: int64


## Step 3: Exemplar Selection

In [29]:
CHUNK_SIZE = 20
PICKS_PER_CHUNK = 5
STOP_THRESHOLD = 25
FINAL_TARGET = 10


def fmt_overall_text(row):
    parts = []
    if pd.notna(row["ECONOMIC_IMPACT_EXPECTATION"]) and row["ECONOMIC_IMPACT_EXPECTATION"]:
        parts.append(f"[Economic] {row['ECONOMIC_IMPACT_EXPECTATION'].strip()}")
    if pd.notna(row["PERSONAL_AI_IMPACT"]) and row["PERSONAL_AI_IMPACT"]:
        parts.append(f"[Personal] {row['PERSONAL_AI_IMPACT'].strip()}")
    return "\n".join(parts)


labeled_df["overall_text"] = labeled_df.apply(fmt_overall_text, axis=1)
labeled_df["government_text"] = labeled_df["GOVERNMENT_ACTION_SUGGESTION"].fillna("").str.strip()

In [30]:
EXEMPLAR_SYSTEM_PROMPT = """You are selecting canonical examples of a labeled survey response for use as few-shot classification examples.

The responses below are all labeled "{label}" for the category "{category}". Select the {n} best examples.

Prefer:
- Substantive over short or obvious: pick responses that make specific arguments or show a clear policy position
- Diverse: if possible, cover different angles of the "{label}" position rather than picking responses that repeat the same point

Return ONLY a JSON array of the response numbers you selected. Example: [2, 7, 14]"""


def cortex_complete(messages, max_tokens=150):
    # Parameterized binding lets the connector handle escaping — avoids Snowflake
    # interpreting \n in json.dumps output as a literal newline before PARSE_JSON sees it.
    messages_json = json.dumps(messages)
    sql = """SELECT SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        PARSE_JSON(%s),
        OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', %s)
    )"""
    cur = conn.cursor()
    cur.execute(sql, [MODEL, messages_json, max_tokens])
    raw = cur.fetchone()[0]
    content = json.loads(raw)["choices"][0]["messages"].strip()
    return re.sub(r"^```(?:json)?\s*", "", content).rstrip("` \n")


def select_from_chunk(texts, label, category, n_picks):
    numbered = "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))
    system = EXEMPLAR_SYSTEM_PROMPT.format(label=label, category=category, n=n_picks)
    content = cortex_complete([
        {"role": "system", "content": system},
        {"role": "user", "content": numbered},
    ])
    try:
        picks = json.loads(content)
        return [p - 1 for p in picks if isinstance(p, int) and 1 <= p <= len(texts)]
    except Exception:
        print(f"  Warning: could not parse picks response: {content[:100]}")
        return list(range(min(n_picks, len(texts))))


def reduce_to_exemplars(texts, label, category):
    pool = list(range(len(texts)))

    while len(pool) > STOP_THRESHOLD:
        next_pool = []
        pool_texts = [texts[i] for i in pool]
        for start in range(0, len(pool_texts), CHUNK_SIZE):
            chunk = pool_texts[start:start + CHUNK_SIZE]
            picks = select_from_chunk(chunk, label, category, min(PICKS_PER_CHUNK, len(chunk)))
            next_pool.extend(pool[start + p] for p in picks)
        pool = next_pool
        print(f"  [{category} / {label}] pool → {len(pool)}")

    if len(pool) > FINAL_TARGET:
        pool_texts = [texts[i] for i in pool]
        picks = select_from_chunk(pool_texts, label, category, FINAL_TARGET)
        pool = [pool[p] for p in picks]

    return pool

In [31]:
CATEGORY_TEXT_COLS = {
    "overall_ai_sentiment": "overall_text",
    "government_action_sentiment": "government_text",
}

exemplar_dfs = {}

for category, text_col in CATEGORY_TEXT_COLS.items():
    exemplar_dfs[category] = {}
    for label in [s.lower() for s in SENTIMENT_OPTIONS]:
        mask = (labeled_df[category] == label) & (labeled_df[text_col] != "")
        subset = labeled_df[mask].copy()
        print(f"\n{category} / {label}: {len(subset)} responses")
        selected = reduce_to_exemplars(subset[text_col].tolist(), label, category)
        exemplar_dfs[category][label] = subset.iloc[selected].reset_index(drop=True)
        print(f"  → {len(selected)} exemplars selected")

print("\nDone!")


overall_ai_sentiment / pro: 177 responses
  [overall_ai_sentiment / pro] pool → 45
  [overall_ai_sentiment / pro] pool → 15
  → 10 exemplars selected

overall_ai_sentiment / anti: 646 responses
  [overall_ai_sentiment / anti] pool → 165
  [overall_ai_sentiment / anti] pool → 45
  [overall_ai_sentiment / anti] pool → 15
  → 10 exemplars selected

overall_ai_sentiment / neutral: 297 responses
  [overall_ai_sentiment / neutral] pool → 75
  [overall_ai_sentiment / neutral] pool → 20
  → 10 exemplars selected

government_action_sentiment / pro: 963 responses
  [government_action_sentiment / pro] pool → 243
  [government_action_sentiment / pro] pool → 63
  [government_action_sentiment / pro] pool → 18
  → 10 exemplars selected

government_action_sentiment / anti: 48 responses
  [government_action_sentiment / anti] pool → 15
  → 10 exemplars selected

government_action_sentiment / neutral: 65 responses
  [government_action_sentiment / neutral] pool → 20
  → 10 exemplars selected

Done!


In [32]:
for category, text_col in CATEGORY_TEXT_COLS.items():
    print(f"\n{'='*70}")
    print(f"CATEGORY: {category.upper()}")
    for label in [s.lower() for s in SENTIMENT_OPTIONS]:
        print(f"\n{'─'*40}\n  LABEL: {label}\n{'─'*40}")
        for rank, row in enumerate(exemplar_dfs[category][label].itertuples(), 1):
            text = getattr(row, text_col)
            print(f"\n[{rank}] {text[:600]}{'...' if len(text) > 600 else ''}")


CATEGORY: OVERALL_AI_SENTIMENT

────────────────────────────────────────
  LABEL: pro
────────────────────────────────────────

[1] [Economic] AI is likely to have a major impact on the economy by increasing productivity, changing how work is performed, and creating new industries, tools, and business models. In many sectors, AI can help organizations process information faster, automate repetitive tasks, improve customer service, reduce administrative burden, and support better decision-making.

For government and public services, AI could improve permitting, benefits administration, public inquiries, fraud detection, cybersecurity, emergency response, and data analysis. For businesses, it may reduce costs, speed up produ...

[2] [Economic] AI is a disruptive technology. People and jobs will be displaced. The more useful question is whether displacement is the same thing as replacement — and history suggests it isn't.

When photography devastated the market for portrait painters, the

In [33]:
keep_cols = [
    "SURVEY_RESPONDENT_ID",
    "ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT",
    "GOVERNMENT_ACTION_SUGGESTION",
    "overall_ai_sentiment", "overall_rationale",
    "government_action_sentiment", "government_rationale",
]

parts = []
for category, text_col in CATEGORY_TEXT_COLS.items():
    for label, sub_df in exemplar_dfs[category].items():
        chunk = sub_df[keep_cols].copy()
        chunk["exemplar_category"] = category
        chunk["exemplar_label"] = label
        chunk["exemplar_text"] = sub_df[text_col].values
        parts.append(chunk)

exemplars_df = pd.concat(parts, ignore_index=True)
exemplars_df.to_csv("exemplars.csv", index=False)
print(f"Saved {len(exemplars_df)} rows to exemplars.csv")